# VLM Assembly Monitor Colab Notebook
**Multimodal VLM for Assembly State Estimation**

This notebook runs the complete pipeline:
1. Environment setup
2. Synthetic dataset generation
3. Model architecture inspection
4. Training with mixed precision
5. Evaluation & attention visualisation
6. ONNX export & edge benchmarking

**Runtime:** Set to GPU (T4) in Edit → Notebook settings

## 1. Environment Setup

In [ ]:
# Clone or mount your repo. For Colab, upload the vlm_assembly/ folder to Drive
# then mount and add to path:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/vlm_assembly')

# Alternatively, clone from your GitHub:
# !git clone https://github.com/YOUR_USERNAME/vlm-assembly-monitor.git
# sys.path.insert(0, '/content/vlm-assembly-monitor')

In [ ]:
!pip install -q torch torchvision timm transformers einops albumentations \
    onnx onnxsim onnxruntime filterpy tqdm wandb pycocotools

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Synthetic Dataset — Quick Data Inspection

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from data.synthetic import SyntheticAssemblyDataset, collate_fn, STATE_LABELS

dataset = SyntheticAssemblyDataset(num_clips=500, T=8, num_objects=3)
sample = dataset[0]

print(f'Clip shape:    {sample["clip"].shape}       # (T, 3, H, W)')
print(f'State label:   {sample["state_label"]} ({STATE_LABELS[sample["state_label"]]}')
print(f'Instruction:   {sample["instruction"]}')
print(f'Boxes shape:   {sample["boxes"].shape}    # (T, max_objects, 4)')

# Visualise first 4 frames with ground truth boxes
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f'State: {STATE_LABELS[sample["state_label"]]} | "{sample["instruction"][:60]}..."', fontsize=10)

for t, ax in enumerate(axes):
    frame = sample['clip'][t].permute(1, 2, 0).numpy()  # (H, W, 3)
    frame = (frame * STD + MEAN).clip(0, 1)
    ax.imshow(frame)
    ax.set_title(f'Frame {t+1}', fontsize=9)
    ax.axis('off')

    H, W = frame.shape[:2]
    for k in range(sample['boxes'].shape[1]):
        if sample['box_mask'][t, k]:
            cx, cy, bw, bh = sample['boxes'][t, k].numpy()
            x1 = (cx - bw/2) * W
            y1 = (cy - bh/2) * H
            rect = patches.Rectangle((x1, y1), bw*W, bh*H,
                                      linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-4, f'P{k+1}', color='lime', fontsize=8)

plt.tight_layout()
plt.savefig('sample_clip.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_clip.png')

## 3. Model Architecture Inspection

In [ ]:
from models.vlm import VLMAssemblyMonitor

model = VLMAssemblyMonitor(
    d_model=512,
    nhead=8,
    num_states=6,
    max_objects=4,
    T_max=8,
    freeze_vision_backbone=True,
    freeze_bert=True,
).to(DEVICE)

param_counts = model.count_parameters()
print('\nTrainable parameter counts:')
print('-' * 40)
for k, v in param_counts.items():
    bar = '█' * (v // 100_000)
    print(f'  {k:25s}: {v:>10,}  {bar}')

# Quick forward pass sanity check
B, T = 2, 8
dummy_clip = torch.randn(B, T, 3, 224, 224).to(DEVICE)
input_ids, attn_mask = model.encode_instruction(
    ['Attach the table leg to the base', 'Tighten the corner bolts'], DEVICE
)

with torch.no_grad():
    outputs = model(dummy_clip, input_ids, attn_mask)

print('\nForward pass output shapes:')
for k, v in outputs.items():
    if v is not None:
        print(f'  {k:20s}: {tuple(v.shape)}')

## 4. Training (Synthetic Data)

In [ ]:
# Run training from the command line (recommended — captures all logs cleanly)
!python train.py \
    --synthetic \
    --num_clips 2000 \
    --epochs 25 \
    --batch_size 4 \
    --grad_accum 4 \
    --T 8 \
    --d_model 512 \
    --freeze_vis \
    --freeze_bert \
    --save_dir /content/checkpoints \
    --no_wandb

## 5. Evaluation & Attention Map Visualisation

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from models.vlm import VLMAssemblyMonitor
from data.synthetic import SyntheticAssemblyDataset, STATE_LABELS
from utils.kalman_filter import BBoxKalmanFilter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

# Load best checkpoint
model = VLMAssemblyMonitor(d_model=512, T_max=8, freeze_vision_backbone=True, freeze_bert=True).to(DEVICE)
ckpt = torch.load('/content/checkpoints/best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print('Model loaded.')

# Pick a test sample
dataset = SyntheticAssemblyDataset(num_clips=200, T=8, num_objects=3)
sample = dataset[42]

clip = sample['clip'].unsqueeze(0).to(DEVICE)        # (1, T, 3, H, W)
input_ids, attn_mask = model.encode_instruction([sample['instruction']], DEVICE)

with torch.no_grad():
    outputs = model(clip, input_ids, attn_mask)

pred_state = outputs['state_logits'].argmax(dim=1).item()
gt_state   = sample['state_label'].item()
print(f'Predicted state: {STATE_LABELS[pred_state]} (GT: {STATE_LABELS[gt_state]})')

# ── Kalman Filter tracking ────────────────────────────────────────────────
tracker = BBoxKalmanFilter()
tracked_per_frame = []

pred_boxes = outputs['boxes'][0].cpu().numpy()       # (T, max_objects, 4)
pred_obj   = torch.sigmoid(outputs['objectness'][0]).cpu().numpy()  # (T, max_objects)

for t in range(pred_boxes.shape[0]):
    dets = np.concatenate([pred_boxes[t], pred_obj[t:t+1].T], axis=1)  # (K, 5)
    tracked = tracker.update(dets, objectness_threshold=0.4)
    tracked_per_frame.append(tracked)

# ── Visualise: predicted boxes + Kalman-smoothed tracks ──────────────────
T_frames = pred_boxes.shape[0]
fig, axes = plt.subplots(2, T_frames // 2, figsize=(T_frames * 2.5, 6))
axes = axes.flatten()

for t in range(T_frames):
    ax = axes[t]
    frame = sample['clip'][t].permute(1,2,0).numpy()
    frame = (frame * STD + MEAN).clip(0, 1)
    H, W = frame.shape[:2]
    ax.imshow(frame)
    ax.set_title(f'Frame {t+1}', fontsize=8)
    ax.axis('off')

    # Raw predictions (red)
    for k in range(pred_boxes.shape[1]):
        if pred_obj[t, k] > 0.4:
            cx, cy, bw, bh = pred_boxes[t, k]
            rect = patches.Rectangle(
                ((cx-bw/2)*W, (cy-bh/2)*H), bw*W, bh*H,
                linewidth=1.5, edgecolor='red', facecolor='none', linestyle='--'
            )
            ax.add_patch(rect)

    # Kalman-smoothed (green)
    for trk in tracked_per_frame[t]:
        if trk['confirmed']:
            cx, cy, bw, bh = trk['box']
            rect = patches.Rectangle(
                ((cx-bw/2)*W, (cy-bh/2)*H), bw*W, bh*H,
                linewidth=2, edgecolor='lime', facecolor='none'
            )
            ax.add_patch(rect)
            ax.text((cx-bw/2)*W+2, (cy-bh/2)*H-4, f'ID{trk["track_id"]}',
                    color='lime', fontsize=7)

fig.suptitle(
    f'Predicted: {STATE_LABELS[pred_state]} | GT: {STATE_LABELS[gt_state]}\n'
    f'Red dashed = raw VLM output | Green = Kalman-smoothed tracks',
    fontsize=10
)
plt.tight_layout()
plt.savefig('tracking_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: tracking_result.png')

## 6. Cross-Attention Map Visualisation

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# attn_weights shape: (B, nhead, L, N)
# L = language tokens, N = spatial visual tokens (7x7=49)
attn = outputs['attn_weights'][0]   # (nhead, L, N)
attn_mean = attn.mean(0).cpu().numpy()  # average over heads: (L, N)

# Decode instruction tokens for x-axis labels
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())
tokens = [t for t in tokens if t not in ['[PAD]']]

# Visualise: for each language token, show the 7x7 spatial attention map
n_show = min(6, len(tokens))
fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.5, 2.5))
for i, ax in enumerate(axes):
    spatial_map = attn_mean[i, :49].reshape(7, 7)  # N=49 → 7x7 grid
    ax.imshow(spatial_map, cmap='hot', vmin=0)
    ax.set_title(tokens[i], fontsize=9)
    ax.axis('off')

fig.suptitle('Cross-attention: which image patches does each word attend to?', fontsize=10)
plt.tight_layout()
plt.savefig('attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: attention_maps.png')

## 7. ONNX Export & Edge Benchmarking

In [ ]:
!python export_onnx.py \
    --checkpoint /content/checkpoints/best.pth \
    --synthetic \
    --output_dir /content/exported_models \
    --batch_size 1 \
    --bench_iters 50 \
    --T 8

In [ ]:
# Display the benchmark table as a styled DataFrame
import json, pandas as pd
with open('/content/exported_models/benchmark_results.json') as f:
    results = json.load(f)

df = pd.DataFrame(results)
df.columns = ['Precision', 'Framework', 'Memory (MB)', 'Latency p50 (ms)', 'Latency p95 (ms)']
print('\n=== EDGE DEPLOYMENT BENCHMARK ===')
print(df.to_string(index=False))
print('\nCopy this table into your README.md!')